```
# Program: DuET v1.1.0
# Author: Sungho Lee, Jae-Won Lee
# Affiliation: MOGAM Institute for Biomedical Research
# Contact: https://github.com/mogam-ai/DuET/issues
# Citation: TBD
```

# DuET Downstream Analysis: Attribution & Attention

Per-fold out-of-fold (OOF) attribution (Integrated Gradients & DeepLIFT-SHAP) and
AttentionPooling analysis. Leakage-free: each fold model attributes only its
held-out test partition; results are concatenated across all 10 folds.


## 1. Model & Data Loading

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from glob import glob
from os.path import getmtime
from tqdm import tqdm
from scipy.signal import savgol_filter
from torch.utils.data import DataLoader

from duet.models.module import Module
from duet.data.datamodule import DataModule
from duet.configs.config import load_cfgs

In [ ]:
# Fold setup (leakage-free OOF). Each fold model -> its own held-out test partition.
# Trained-model run dirs: _logs/<exp_name>-...-split{00..09}. Set to your run.
EXP_PREFIX = "_logs/duet-utr5.500+cds.3000-seed42-split"
N_FOLDS = 10

# Attribution target: which of the 76 cell-type outputs to attribute.
#   None  -> mean over all 76 cell types (closest to single-task pan-celltype model)
#   str   -> a cell-type name, e.g. 'all-celltype' or 'a549' (TE_ prefix optional)
#   int   -> column index into the 76 TE_ columns
TARGET_CELLTYPE = None

ATTR_METHOD = "shap"   # "shap" (DeepLiftShap, dinuc baseline) or "ig"
assert ATTR_METHOD in ("shap", "ig")

# Resolve cell-type list + target index
import pandas as _pd
ref_dir = f"{EXP_PREFIX}00"
cfg, dict_cfg = load_cfgs([f"{ref_dir}/config.yaml"], {})
_te_cols = [c for c in _pd.read_csv(cfg.dataset.test, sep='\t', nrows=1).columns if c.startswith('TE_')]
CELLTYPES = [c[3:] for c in _te_cols]   # strip 'TE_'
assert len(CELLTYPES) == cfg.model.param.n_targets, (len(CELLTYPES), cfg.model.param.n_targets)
if TARGET_CELLTYPE is None:
    TARGET_IDX = None
    _tname = 'MEAN(76 cell types)'
elif isinstance(TARGET_CELLTYPE, int):
    TARGET_IDX = TARGET_CELLTYPE; _tname = CELLTYPES[TARGET_IDX]
else:
    _q = TARGET_CELLTYPE[3:] if TARGET_CELLTYPE.startswith('TE_') else TARGET_CELLTYPE
    TARGET_IDX = CELLTYPES.index(_q); _tname = CELLTYPES[TARGET_IDX]

print(f"Model: {cfg.model.name} (n_targets={cfg.model.param.n_targets}), dataset: {cfg.dataset.test}")
print(f"KFold: k={cfg.datamodule.kfold_test_k}, seed={cfg.datamodule.kfold_test_seed}")
print(f"Attribution method: {ATTR_METHOD} | target: {_tname} (idx={TARGET_IDX})")
print(f"Available cell types ({len(CELLTYPES)}): {CELLTYPES[:6]} ...")


def scalar_labels(labels):
    """Reduce multitask labels (N,76, NaN-masked) to a scalar per transcript,
    consistent with the attribution TARGET_IDX. Single-task labels pass through."""
    import numpy as _np
    labels = _np.asarray(labels)
    if labels.ndim == 2 and labels.shape[1] > 1:
        return _np.nanmean(labels, axis=1) if TARGET_IDX is None else labels[:, TARGET_IDX]
    return labels


In [ ]:
# Build full dataset once, then reproduce the exact KFold test partition per fold.
from sklearn.model_selection import KFold

cfg.datamodule.do_kfold_test = False  # we slice folds manually below
scaler_path = f"{ref_dir}/label_scaler.joblib"
scaler_obj = joblib.load(scaler_path) if os.path.exists(scaler_path) else None

datamodule = DataModule(cfg, cfg.dataset.test, scaler_obj=scaler_obj)
datamodule.setup("test")
full_dataset = datamodule.test_dataset   # full dataset (do_kfold_test=False)
print(f"Full dataset size: {len(full_dataset)}")

# Reproduce training KFold split (KFold depends only on n_rows + seed + k; row order unchanged).
kf = KFold(n_splits=cfg.datamodule.kfold_test_k, shuffle=True,
           random_state=cfg.datamodule.kfold_test_seed)
fold_test_idx = [test_idx for _, test_idx in kf.split(datamodule.dataset.data)]
assert sum(len(t) for t in fold_test_idx) == len(full_dataset)
print("Per-fold test sizes:", [len(t) for t in fold_test_idx])

## 2. Dinucleotide Shuffle Baseline Generation

In [ ]:
def dinucleotide_shuffle(seq: str, rng=None) -> str:
    """Shuffle sequence preserving dinucleotide frequencies (Altschul-Erickson algorithm)."""
    if rng is None:
        rng = np.random.default_rng()
    seq = list(seq)
    # Build edge list for each dinucleotide
    from collections import defaultdict
    edges = defaultdict(list)
    for i in range(len(seq) - 1):
        edges[seq[i]].append(i + 1)
    # Shuffle edges for each starting nucleotide
    for k in edges:
        rng.shuffle(edges[k])
    # Euler path reconstruction
    result = [seq[0]]
    last_idx = 0
    for _ in range(len(seq) - 1):
        nxt_list = edges[seq[last_idx]]
        if not nxt_list:
            break
        nxt_idx = nxt_list.pop()
        result.append(seq[nxt_idx])
        last_idx = nxt_idx
    return "".join(result)


def generate_shuffle_baselines(dataset, n_baselines=20, seed=42):
    """Generate dinucleotide-shuffled baseline sequences from dataset samples."""
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(n_baselines, len(dataset)), replace=False)
    
    baselines_utr = []
    baselines_cds = []
    
    for idx in indices:
        item = dataset[int(idx)]
        # Get original sequences from dataset
        utr_onehot = item["utr"].numpy()  # (utr_len, 4)
        cds_onehot = item["cds"].numpy()  # (cds_len, 64)
        
        # Reconstruct UTR sequence from one-hot, shuffle, re-encode
        utr_seq = "".join(["AGCT"[c] if c < 4 else "N" for c in utr_onehot.argmax(axis=-1)])
        utr_shuffled = dinucleotide_shuffle(utr_seq.replace("N", ""), rng)
        utr_shuffled = "N" * (len(utr_seq) - len(utr_shuffled)) + utr_shuffled  # re-pad left
        
        # For CDS (codon-level): shuffle at codon level preserving dicodon frequencies
        # Simplified: just shuffle codons randomly (true dinucleotide shuffle at nt level is complex for codons)
        cds_indices = cds_onehot.argmax(axis=-1)  # (cds_len,)
        non_pad = cds_indices[cds_indices != 64]
        rng.shuffle(non_pad)
        cds_shuffled = np.full_like(cds_indices, 64)
        cds_shuffled[:len(non_pad)] = non_pad
        
        # Re-encode to one-hot
        utr_enc = np.zeros_like(utr_onehot)
        for i, nt in enumerate(utr_shuffled):
            if nt in "ACGT":
                utr_enc[i, "AGCT".index(nt)] = 1.0
        
        cds_enc = np.zeros_like(cds_onehot)
        for i, c in enumerate(cds_shuffled):
            if c < 64:
                cds_enc[i, c] = 1.0
        
        baselines_utr.append(utr_enc)
        baselines_cds.append(cds_enc)
    
    return np.array(baselines_utr, dtype=np.float32), np.array(baselines_cds, dtype=np.float32)


baselines_utr, baselines_cds = generate_shuffle_baselines(datamodule.test_dataset, n_baselines=20)
print(f"Baselines: UTR {baselines_utr.shape}, CDS {baselines_cds.shape}")


## 3. Attribution Computation

In [ ]:
from captum.attr import LayerIntegratedGradients, LayerDeepLiftShap


class DualBranchForward(torch.nn.Module):
    """Wrapper that takes (utr_onehot, cds_onehot) and returns scalar TE prediction.
    Bypasses argmax by using matmul with embedding weights."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, utr_onehot, cds_onehot):
        utr_vocab = self.model.utr_branch.embedding.weight.shape[0]  # 5
        cds_vocab = self.model.cds_branch.embedding.weight.shape[0]  # 65
        utr_padded = torch.nn.functional.pad(utr_onehot.float(), (0, utr_vocab - utr_onehot.shape[-1]))
        cds_padded = torch.nn.functional.pad(cds_onehot.float(), (0, cds_vocab - cds_onehot.shape[-1]))
        utr_emb = utr_padded @ self.model.utr_branch.embedding.weight
        cds_emb = cds_padded @ self.model.cds_branch.embedding.weight

        utr_out = self._branch_forward(self.model.utr_branch, utr_emb, utr_onehot)
        cds_out = self._branch_forward(self.model.cds_branch, cds_emb, cds_onehot)

        features = [utr_out, cds_out]
        if self.model.use_start_context and "start" in self._current_batch:
            start_onehot = self._current_batch["start"].float()
            start_emb = start_onehot @ self.model.tis_branch.embedding.weight
            start_out = self._branch_forward(self.model.tis_branch, start_emb, start_onehot)
            features.append(start_out)
        if self.model.use_sequence_feature and "sequence_feature" in self._current_batch:
            features.append(self.model.feat_proj(self._current_batch["sequence_feature"]))

        out = self.model.head(torch.cat(features, dim=1))  # (B,) single or (B,76) multitask
        if out.dim() == 2 and out.shape[1] > 1:
            # multitask: reduce 76 outputs to a scalar per sample for attribution
            if TARGET_IDX is None:
                out = out.mean(dim=1)
            else:
                out = out[:, TARGET_IDX]
        else:
            out = out.squeeze(-1)
        return out

    def _branch_forward(self, branch, emb, onehot):
        """Run branch from embedding output onwards.
        Branch: initial_conv -> res_blocks -> BiGRU -> CNN skip-concat -> attn_pool."""
        mask = onehot.sum(dim=-1) > 0  # non-padding mask
        x = emb.transpose(1, 2)  # (B, embed_dim, L)
        x = branch.initial_conv(x)
        for block in branch.res_blocks:
            x = block(x)
        cnn_out = x.transpose(1, 2)              # (B, L, cnn_filters)
        gru_out, _ = branch.gru(cnn_out)         # (B, L, gru_hidden*2)
        combined = torch.cat([cnn_out, gru_out], dim=-1)  # v5 CNN skip-concat
        return branch.attn_pool(combined, mask)

    _current_batch = {}


def load_fold_wrapper(fold):
    """Load fold model checkpoint and wrap it. train() mode for cuDNN RNN DeepLift backward."""
    exp_dir = f"{EXP_PREFIX}{fold:02d}"
    ckpt = sorted(glob(f"{exp_dir}/ckpts/*.ckpt"), key=getmtime)[-1]
    fcfg, fdict = load_cfgs([f"{exp_dir}/config.yaml"], {})
    m = Module.load_from_checkpoint(ckpt, cfg=fcfg, dict_cfg=fdict, strict=False)
    m.eval(); m.cuda()
    w = DualBranchForward(m.model)
    w.train(); w.cuda()
    return w


# Smoke test: fold 0 wrapper on its first held-out batch
_w0 = load_fold_wrapper(0)
_loader0 = DataLoader(torch.utils.data.Subset(full_dataset, fold_test_idx[0]),
                      batch_size=16, shuffle=False, num_workers=1)
_sb = next(iter(_loader0))
_w0._current_batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in _sb.items()}
with torch.no_grad():
    _to = _w0(_sb["utr"].cuda(), _sb["cds"].cuda())
print(f"Fold-0 wrapper forward OK: output shape {_to.shape}")
del _w0, _loader0; gc_ok = True

In [ ]:
def compute_layer_attribution(data_loader, wrapper, baselines_utr, baselines_cds, method="ig", n_steps=50):
    """Compute attribution scores using LayerIG or LayerDeepLiftShap.
    
    Args:
        method: "ig" for LayerIntegratedGradients, "shap" for LayerDeepLiftShap
    Returns:
        dict with utr_attr (N, utr_len, utr_embed), cds_attr (N, cds_len, cds_embed), labels (N,)
    """
    # For Layer methods we need the embedding layers
    # But since we bypass embedding via matmul, we attribute to the input directly
    # Use regular (non-Layer) IG/DeepLiftShap on the wrapper inputs
    if method == "ig":
        from captum.attr import IntegratedGradients
        attr_method = IntegratedGradients(wrapper)
    else:
        from captum.attr import DeepLiftShap
        attr_method = DeepLiftShap(wrapper)
    
    baselines_utr_t = torch.tensor(baselines_utr).cuda()
    baselines_cds_t = torch.tensor(baselines_cds).cuda()
    
    all_utr_attr, all_cds_attr, all_labels = [], [], []
    
    for batch in tqdm(data_loader, desc=f"Attribution ({method})"):
        utr = batch["utr"].cuda().float()
        cds = batch["cds"].cuda().float()
        wrapper._current_batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        if method == "ig":
            # Use zero baseline for IG (padding = no sequence)
            utr_baseline = torch.zeros_like(utr)
            cds_baseline = torch.zeros_like(cds)
            attr_utr, attr_cds = attr_method.attribute(
                (utr, cds), baselines=(utr_baseline, cds_baseline),
                n_steps=n_steps, target=None)
        else:
            # DeepLiftShap: expand baselines for batch
            attr_utr, attr_cds = attr_method.attribute(
                (utr, cds), baselines=(baselines_utr_t, baselines_cds_t),
                target=None)
        
        all_utr_attr.append(attr_utr.detach().cpu().numpy())
        all_cds_attr.append(attr_cds.detach().cpu().numpy())
        all_labels.append(batch["y"].numpy())
    
    return {
        "utr_attr": np.concatenate(all_utr_attr),   # (N, utr_len, utr_channels)
        "cds_attr": np.concatenate(all_cds_attr),   # (N, cds_len, cds_channels)
        "labels": np.concatenate(all_labels),
    }


In [ ]:
# Per-fold OOF attribution (single method = ATTR_METHOD). Each fold model attributes
# only its held-out test set; results are concatenated (full dataset covered once).
import gc
output_dir = Path(f"{ref_dir}/attribution_oof_multitask")
output_dir.mkdir(exist_ok=True)

fold_attrs = []
for fold in range(N_FOLDS):
    print(f"--- Fold {fold} ({ATTR_METHOD}) ---")
    wrapper = load_fold_wrapper(fold)
    test_subset = torch.utils.data.Subset(full_dataset, fold_test_idx[fold])
    loader = DataLoader(test_subset, batch_size=16, shuffle=False, num_workers=1)
    if ATTR_METHOD == "shap":
        # Fold-specific dinucleotide-shuffle baselines (from this fold's held-out set)
        b_utr, b_cds = generate_shuffle_baselines(test_subset, n_baselines=20, seed=42 + fold)
        fa = compute_layer_attribution(loader, wrapper, b_utr, b_cds, method="shap")
    else:
        fa = compute_layer_attribution(loader, wrapper, None, None, method="ig", n_steps=50)
    fold_attrs.append(fa)
    del wrapper; gc.collect(); torch.cuda.empty_cache()

# attr_shap holds the chosen-method result (variable name kept for downstream cells)
attr_shap = {k: np.concatenate([fa[k] for fa in fold_attrs]) for k in fold_attrs[0]}
for key, arr in attr_shap.items():
    np.save(output_dir / f"{ATTR_METHOD}_{key}.npy", arr)
print(f"OOF {ATTR_METHOD} saved. UTR attr: {attr_shap['utr_attr'].shape}, "
      f"CDS attr: {attr_shap['cds_attr'].shape}")

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

## TF-MoDISco export

Export one-hot sequences + SHAP attributions in TF-MoDISco-lite format `(N, 4, L)`.
Reuses the OOF `attr_shap` already computed/saved above (no GPU re-inference).
One-hot is rebuilt from `full_dataset` in the **same OOF concatenation order** as the attributions
(fold 0, fold 1, ... concatenated), so sequence[i] aligns with attribution[i].

UTR branch only (channels A,C,G,T). CDS is 64-codon space and needs the noembed/4-ch model for
nucleotide-level modisco, so it is skipped here.

Run `modisco motifs -s modisco_utr_seq.npy -a modisco_utr_attr.npy -n 50000 -o modisco_results.h5`.

In [ ]:
# --- TF-MoDISco-lite export (reuses saved OOF attr_shap; rebuilds one-hot, no GPU) ---
import numpy as np
from pathlib import Path

# Output filename suffix separating single vs multitask modisco inputs.
MODISCO_SUFFIX = '_multitask'
modisco_dir = Path('motif_search/data')
modisco_dir.mkdir(parents=True, exist_ok=True)

# Load saved OOF attribution if attr_shap not in memory (avoids recompute).
try:
    attr_shap
except NameError:
    out = Path(ref_dir) / 'attribution_oof'
    attr_shap = {
        'utr_attr': np.load(out / f'{ATTR_METHOD}_utr_attr.npy'),
        'cds_attr': np.load(out / f'{ATTR_METHOD}_cds_attr.npy'),
        'labels':   np.load(out / f'{ATTR_METHOD}_labels.npy'),
    }

utr_attr = attr_shap['utr_attr']  # (N, utr_len, 4), OOF-concatenated order
N, L, C = utr_attr.shape
assert C == 4, f'expected 4 UTR channels, got {C}'

# Rebuild UTR one-hot in the SAME OOF order (fold0,fold1,... concat) as attributions.
oof_order = np.concatenate(fold_test_idx)  # global indices in attribution order
seq_onehot = np.zeros((N, L, 4), dtype=np.float32)
for row, gidx in enumerate(oof_order):
    item = full_dataset[int(gidx)]
    u = item['utr']
    u = u.numpy() if hasattr(u, 'numpy') else np.asarray(u)
    if u.shape[0] == 4 and u.shape[-1] != 4:  # (4,L) -> (L,4)
        u = u.T
    seq_onehot[row] = u[:, :4]

# Sanity: attribution should be ~0 at padding positions (one-hot all-zero rows).
pad_mask = seq_onehot.sum(-1) == 0
print(f'N={N} L={L}  padding fraction={pad_mask.mean():.3f}')
print(f'mean |attr| at non-pad={np.abs(utr_attr[~pad_mask]).mean():.4g}, '
      f'at pad={np.abs(utr_attr[pad_mask]).mean():.4g}')

# TF-MoDISco-lite expects (N, 4, L). Transpose both.
seq_NCL  = np.ascontiguousarray(seq_onehot.transpose(0, 2, 1))   # (N,4,L)
attr_NCL = np.ascontiguousarray(utr_attr.transpose(0, 2, 1))     # (N,4,L)

np.save(modisco_dir / f'modisco_utr_seq_{ATTR_METHOD}{MODISCO_SUFFIX}.npy',  seq_NCL)
np.save(modisco_dir / f'modisco_utr_attr_{ATTR_METHOD}{MODISCO_SUFFIX}.npy', attr_NCL)
print('saved:', modisco_dir / f'modisco_utr_seq_{ATTR_METHOD}{MODISCO_SUFFIX}.npy', seq_NCL.shape)
print('saved:', modisco_dir / f'modisco_utr_attr_{ATTR_METHOD}{MODISCO_SUFFIX}.npy', attr_NCL.shape)
print('\\nRun:')
print(f'  conda activate motif')
print(f'  modisco motifs -s motif_search/data/modisco_utr_seq_{ATTR_METHOD}{MODISCO_SUFFIX}.npy \\\\')
print(f'                 -a motif_search/data/modisco_utr_attr_{ATTR_METHOD}{MODISCO_SUFFIX}.npy \\\\')
print(f'                 -n 50000 -w 500 -o motif_search/modisco_utr_{ATTR_METHOD}{MODISCO_SUFFIX}.h5 -v')


## 4. Position-wise Score Aggregation

In [ ]:
def aggregate_position_scores(attr_dict, method_name=""):
    """Aggregate attribution to position-level scores.

    Returns dict with:
        utr_norm: (N, utr_len) - L2 norm across embedding dim (unsigned importance)
        utr_sum: (N, utr_len) - sum across embedding dim (signed)
        cds_norm, cds_sum: same for CDS
        utr_mask, cds_mask: padding masks (True = real nucleotide/codon)
    """
    utr_attr = attr_dict["utr_attr"]  # (N, utr_len, channels)
    cds_attr = attr_dict["cds_attr"]  # (N, cds_len, channels)

    # Mask from the original one-hot (not from attribution): SHAP/IG leak tiny
    # attribution onto padding, so treat all-zero one-hot rows as padding.
    _oof = np.concatenate(fold_test_idx)  # global indices, attribution order
    Nu, Lu = utr_attr.shape[0], utr_attr.shape[1]
    Lc = cds_attr.shape[1]
    utr_mask = np.zeros((Nu, Lu), dtype=bool)
    cds_mask = np.zeros((Nu, Lc), dtype=bool)
    for row, gidx in enumerate(_oof):
        item = full_dataset[int(gidx)]
        u = item["utr"]; u = u.numpy() if hasattr(u, "numpy") else np.asarray(u)
        if u.shape[0] != Lu and u.shape[-1] == Lu:  # (C,L) -> (L,C)
            u = u.T
        utr_mask[row] = u.sum(-1) > 0
        if Lc > 0 and "cds" in item and np.asarray(item["cds"]).size:
            c = item["cds"]; c = c.numpy() if hasattr(c, "numpy") else np.asarray(c)
            if c.shape[0] != Lc and c.shape[-1] == Lc:
                c = c.T
            cds_mask[row] = c.sum(-1) > 0

    print(f"  padding-mask (one-hot): UTR valid frac={utr_mask.mean():.3f}, "
          f"CDS valid frac={cds_mask.mean():.3f}")
    return {
        "utr_norm": np.linalg.norm(utr_attr, axis=-1),    # (N, utr_len)
        "utr_sum": utr_attr.sum(axis=-1),                  # (N, utr_len)
        "cds_norm": np.linalg.norm(cds_attr, axis=-1),    # (N, cds_len)
        "cds_sum": cds_attr.sum(axis=-1),                  # (N, cds_len)
        "utr_mask": utr_mask,
        "cds_mask": cds_mask,
        "labels": attr_dict["labels"],
    }


scores_shap = aggregate_position_scores(attr_shap, "DeepLiftShap")
print("Aggregation done.")
print(f"  UTR norm shape: {scores_shap['utr_norm'].shape}")
print(f"  CDS norm shape: {scores_shap['cds_norm'].shape}")


# Keep positions where at least this fraction of transcripts
# are valid (UTR is left-padded / CDS right-padded, so short-transcript
# regions have low coverage). 0.05 shows them; raise toward 0.5 to restrict.
POS_MIN_FRAC = 0.05


## 5. Nucleotide-level Attribution (Channel Decomposition)

In [ ]:
def decompose_to_nucleotide(attr_dict, model):
    """Extract nucleotide-level attribution from input-level attribution.
    
    Since the wrapper operates on one-hot inputs directly (matmul bypass),
    the attribution is already in nucleotide space:
      UTR attr: (N, utr_len, 4) -> channels are A, C, G, T
      CDS attr: (N, cds_len, 64) -> channels are codons
    
    No projection needed - just return the raw attribution per channel.
    """
    return {
        "utr_nt_attr": attr_dict["utr_attr"],          # (N, utr_len, 4) - per nucleotide, signed
        "cds_codon_attr": attr_dict["cds_attr"],       # (N, cds_len, 64) - per codon, signed
    }


nt_attr_shap = decompose_to_nucleotide(attr_shap, None)
print(f"Nucleotide-level UTR: {nt_attr_shap['utr_nt_attr'].shape}")  # (N, utr_len, 4)


## 6. Positional Importance Profile Visualization

In [ ]:
def plot_positional_profile(scores, region="utr", agg="norm", method_name="", smooth=True, xlim=None, save_path=None):
    """Plot mean positional importance profile."""
    key = f"{region}_{agg}"
    mask_key = f"{region}_mask"
    
    values = scores[key]
    mask = scores[mask_key]
    masked_values = np.where(mask, values, np.nan)
    mean_profile = np.nanmean(masked_values, axis=0)

    # 95% CI = mean +/- 1.96 * SEM, SEM = nanstd / sqrt(n_valid) per position.
    n_valid = mask.sum(axis=0).astype(float)
    std_profile = np.nanstd(masked_values, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        sem_profile = std_profile / np.sqrt(np.maximum(n_valid, 1))
    ci_profile = 1.96 * sem_profile

    position_mask = np.nanmean(mask.astype(float), axis=0) > POS_MIN_FRAC
    mean_profile[~position_mask] = np.nan
    ci_profile[~position_mask] = np.nan
    
    L = mean_profile.shape[0]
    x = np.arange(1, L + 1)
    max_pos = 500 if region == "utr" else 1000
    
    fig, ax = plt.subplots(figsize=(14, 5))
    color = "steelblue" if region == "utr" else "coral"

    # draw 95% CI ribbon around the mean (smoothed to match the line).
    _band = ci_profile.copy()
    if smooth:
        _v = ~np.isnan(mean_profile)
        if _v.sum() > 5:
            _w = min(11 if region == "utr" else 31, int(_v.sum()) // 2 * 2 - 1)
            if _w >= 5:
                _sm = np.full_like(mean_profile, np.nan)
                _sm[_v] = savgol_filter(mean_profile[_v], window_length=_w, polyorder=3)
                _sb = np.full_like(_band, np.nan)
                _sb[_v] = savgol_filter(np.nan_to_num(ci_profile[_v]), window_length=_w, polyorder=3)
                _center, _band = _sm, np.abs(_sb)
            else:
                _center = mean_profile
        else:
            _center = mean_profile
    else:
        _center = mean_profile
    ax.fill_between(np.arange(1, mean_profile.shape[0] + 1),
                    _center - _band, _center + _band,
                    color=color, alpha=0.2, linewidth=0, label="95% CI")
    
    if smooth:
        valid = ~np.isnan(mean_profile)
        valid_indices = np.where(valid)[0]
        if len(valid_indices) > 5:
            window = min(11 if region == "utr" else 31, len(valid_indices) // 2 * 2 - 1)
            if window >= 5:
                smoothed = np.full_like(mean_profile, np.nan)
                smoothed[valid] = savgol_filter(mean_profile[valid], window_length=window, polyorder=3)
                ax.plot(x, mean_profile, alpha=0.2, color=color)
                ax.plot(x, smoothed, linewidth=2, color=color)
            else:
                ax.plot(x, mean_profile, linewidth=1.5, color=color)
        else:
            ax.plot(x, mean_profile, linewidth=1.5, color=color)
    else:
        ax.plot(x, mean_profile, linewidth=1.5, color=color)
    
    if region == "utr":
        kozak_start = L - 9 + 1
        ax.axvspan(kozak_start, L, alpha=0.2, color="skyblue", label="Kozak region")
        ax.legend()
    
    ax.set_xlabel("Position")
    ax.set_ylabel("Mean |Attribution| (L2 norm)" if agg == "norm" else "Mean Attribution (sum, signed)")
    region_name = "5' UTR" if region == "utr" else "CDS"
    ax.set_title(f"{region_name} Positional Importance ({method_name})")
    
    # X limits with consistent margins
    from matplotlib.ticker import MultipleLocator, FuncFormatter
    if xlim is not None:
        span = xlim[1] - xlim[0]
        margin = max(span * 0.03, 1)
        ax.set_xlim(max(1 - margin, xlim[0] - margin), min(xlim[1] + margin, max_pos + margin))
        step = max(1, int(span / 20))
    else:
        margin = max_pos * 0.01
        ax.set_xlim(1 - margin, max_pos + margin)
        step = max(1, max_pos // 20)
    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: str(int(v)) if 1 <= v <= max_pos else ""))
    
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


plot_positional_profile(scores_shap, "utr", "norm", "DeepLiftShap", smooth=True, save_path=output_dir / "utr_profile_shap_norm.png")
plot_positional_profile(scores_shap, "cds", "norm", "DeepLiftShap", smooth=True, save_path=output_dir / "cds_profile_shap_norm.png")
plot_positional_profile(scores_shap, "utr", "sum", "DeepLiftShap", smooth=True, save_path=output_dir / "utr_profile_shap_sum.png")



In [ ]:
# Plot for DeepLiftShap
plot_positional_profile(scores_shap, "utr", "norm", "DeepLiftShap", smooth=False, xlim=(400,500))
plot_positional_profile(scores_shap, "cds", "norm", "DeepLiftShap", smooth=False, xlim=(0,100))
plot_positional_profile(scores_shap, "utr", "sum", "DeepLiftShap", smooth=False, xlim=(400,500))


## 7. UTR Base Channel-wise Score

In [ ]:
# UTR base-channel line profile (frequency effect removed).
# average each base channel ONLY over transcripts where that
# base is actually present (present = argmax|attr|), padding excluded. Removes the
# composition bias that made frequent A dominate the raw all-transcript average.
def plot_nucleotide_profile(nt_attr, mask, method_name="", smooth=True, xlim=None, save_path=None):
    """Per-base positional attribution for 5'UTR, frequency-effect removed.

    nt_attr: (N, L, 4) signed attribution (channels A,G,C,T)
    mask:    (N, L) True = real nucleotide (padding excluded)
    For base j at position p, mean is taken over transcripts where base j is the
    present base (argmax|attr|) AND not padding -> per-occurrence positional profile.
    """
    N, L, C = nt_attr.shape
    present = np.abs(nt_attr).argmax(-1)                      # (N, L) present base
    mean_nt = np.full((L, C), np.nan)
    for j in range(C):
        sel = (present == j) & mask                          # (N, L)
        cnt = sel.sum(0)                                     # (L,)
        with np.errstate(invalid="ignore"):
            s = np.where(sel, nt_attr[:, :, j], 0.0).sum(0)
            mean_nt[:, j] = np.where(cnt >= 10, s / np.maximum(cnt, 1), np.nan)

    x = np.arange(1, L + 1)
    max_pos = 500
    nucleotides = ["A", "G", "C", "T"]
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    fig, ax = plt.subplots(figsize=(14, 5))
    for j, (nt, color) in enumerate(zip(nucleotides, colors)):
        profile = mean_nt[:, j]
        valid = ~np.isnan(profile)
        if smooth and valid.sum() > 5:
            window = min(11, int(valid.sum()) // 2 * 2 - 1)
            if window >= 5:
                smoothed = np.full_like(profile, np.nan)
                smoothed[valid] = savgol_filter(profile[valid], window_length=window, polyorder=3)
                ax.plot(x, smoothed, linewidth=1.5, color=color, label=nt, alpha=0.8)
            else:
                ax.plot(x, profile, linewidth=1.5, color=color, label=nt, alpha=0.8)
        else:
            ax.plot(x, profile, linewidth=1.5, color=color, label=nt, alpha=0.8)

    kozak_start = L - 9 + 1
    ax.axvspan(kozak_start, L, alpha=0.2, color="skyblue", label="Kozak region")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_xlabel("Position")
    ax.set_ylabel("Attribution (signed, per occurrence)")
    ax.set_title(f"5' UTR Base Line Profile, frequency-removed ({method_name})")
    ax.legend()

    from matplotlib.ticker import MultipleLocator, FuncFormatter
    if xlim is not None:
        span = xlim[1] - xlim[0]
        margin = max(span * 0.03, 1)
        ax.set_xlim(max(1 - margin, xlim[0] - margin), min(xlim[1] + margin, max_pos + margin))
        step = max(1, int(span / 20))
    else:
        margin = max_pos * 0.01
        ax.set_xlim(1 - margin, max_pos + margin)
        step = max(1, max_pos // 20)
    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: str(int(v)) if 1 <= v <= max_pos else ""))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


plot_nucleotide_profile(nt_attr_shap["utr_nt_attr"], scores_shap["utr_mask"],
                        "DeepLiftShap", smooth=True, save_path=output_dir / "utr_nucleotide_profile_shap.png")



# per-base frequency + per-occurrence sensitivity (padding excluded).
_nt = nt_attr_shap["utr_nt_attr"]; _m = scores_shap["utr_mask"]
_pres = np.abs(_nt).argmax(-1)
_bases = ["A", "G", "C", "T"]
_rows = []
for _j in range(4):
    _sel = (_pres == _j) & _m
    _a = _nt[:, :, _j][_sel]
    _rows.append({"base": _bases[_j], "n_occurrence": int(_sel.sum()),
                  "frequency": _sel.sum() / _m.sum(),
                  "per_occ_signed": float(_a.mean()) if _a.size else np.nan,
                  "per_occ_abs": float(np.abs(_a).mean()) if _a.size else np.nan,
                  "total_abs": float(np.abs(_a).sum())})
_utr_occ = pd.DataFrame(_rows)
_utr_occ["total_abs_share"] = _utr_occ["total_abs"] / _utr_occ["total_abs"].sum()
_utr_occ = _utr_occ.drop(columns="total_abs").sort_values("per_occ_abs", ascending=False).reset_index(drop=True)
_utr_occ.to_csv(output_dir / "utr_base_occurrence_summary.tsv", sep="\t", index=False)
print("saved utr_base_occurrence_summary.tsv")
print(_utr_occ.to_string(index=False))


In [ ]:
plot_nucleotide_profile(nt_attr_shap["utr_nt_attr"], scores_shap["utr_mask"],
                        "DeepLiftShap", smooth=False, xlim=(450,500))

In [ ]:
# Kozak region heatmap: last 14 positions, per-nucleotide attribution
kozak_len = 14
nt_mean = np.nanmean(np.where(scores_shap["utr_mask"][:, :, None], nt_attr_shap["utr_nt_attr"], np.nan), axis=0)
kozak_data = nt_mean[-kozak_len:, :]  # (14, 4)

fig, ax = plt.subplots(figsize=(kozak_len * 0.8, 4))
im = ax.imshow(kozak_data.T, aspect=1, cmap="viridis", interpolation="nearest")

ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["A", "G", "C", "T"])

# X axis: Kozak consensus labels (1-based from end)
x_labels = ["n", "n", "n", "n", "n", "(g,c,c)", "g", "c", "c", "A/G", "c", "c", "A", "U/G"]
# Adjust to 14 positions: positions -14 to -1 relative to AUG
x_labels_14 = ["n", "n", "n", "n", "n", "(g", "c", "c)", "g", "c", "c", "A/G", "c", "c"]
ax.set_xticks(range(kozak_len))
ax.set_xticklabels(x_labels_14, fontsize=9)
ax.set_xlabel("Kozak consensus position")
ax.set_title("Kozak Region Attribution (last 14 nt of 5' UTR)")

# Annotate Kozak consensus nucleotides on last 9 positions (indices 5-13)
# Kozak: (g/c)ccGccAUG -> positions -9 to -1
# Index in kozak_data: 5=g/c, 6=c, 7=c, 8=g, 9=c, 10=c, 11=A/G, 12=c, 13=c
kozak_annotations = {
    5: ["G"],       # g/c
    6: ["C"],            # c
    7: ["C"],            # c
    8: ["G"],            # g
    9: ["C"],            # c
    10: ["C"],           # c
    11: ["A", "G"],      # A/G (Kozak +4 equivalent, or -3 position)
    12: ["C"],           # c
    13: ["C"],           # c
}
nt_to_row = {"A": 0, "C": 2, "G": 1, "T": 3}

for col, nts in kozak_annotations.items():
    for nt in nts:
        row = nt_to_row[nt]
        ax.text(col, row, nt, ha="center", va="center", fontsize=20, fontweight="bold", color="black")

ax.set_xticks(np.arange(-0.5, kozak_len, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=0.5)
ax.tick_params(which="minor", length=0)
plt.colorbar(im, ax=ax, label="Attribution", shrink=0.8)
plt.tight_layout()
plt.savefig(output_dir / "kozak_heatmap.png", dpi=150)
plt.show()


# frequency-removed version: each (position, base) is the
# mean over transcripts where that base is PRESENT (argmax|attr|), padding excluded.
# Comparison to the raw all-transcript heatmap above (which is composition-weighted).
_ku_nt = nt_attr_shap["utr_nt_attr"]; _ku_m = scores_shap["utr_mask"]
_ku_present = np.abs(_ku_nt).argmax(-1)
_L4 = _ku_nt.shape[1]
_koz_adj = np.full((_L4, 4), np.nan)
for _j in range(4):
    _sel = (_ku_present == _j) & _ku_m
    _c = _sel.sum(0)
    with np.errstate(invalid="ignore"):
        _s = np.where(_sel, _ku_nt[:, :, _j], 0.0).sum(0)
        _koz_adj[:, _j] = np.where(_c >= 10, _s / np.maximum(_c, 1), np.nan)
kozak_adj = _koz_adj[-kozak_len:, :]   # (14, 4)

fig, ax = plt.subplots(figsize=(kozak_len * 0.8, 4))
_vmax = np.nanpercentile(np.abs(kozak_adj), 99)
im = ax.imshow(kozak_adj.T, aspect=1, cmap="RdBu_r", vmin=-_vmax, vmax=_vmax, interpolation="nearest")
ax.set_yticks([0, 1, 2, 3]); ax.set_yticklabels(["A", "G", "C", "T"])
ax.set_xticks(range(kozak_len)); ax.set_xticklabels(x_labels_14, fontsize=9)
ax.set_xlabel("Kozak consensus position")
ax.set_title("Kozak Region Attribution, frequency-removed (per-occurrence)")
for col, nts in kozak_annotations.items():
    for nt in nts:
        ax.text(col, nt_to_row[nt], nt, ha="center", va="center", fontsize=20, fontweight="bold", color="black")
ax.set_xticks(np.arange(-0.5, kozak_len, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which="minor", color="gray", linewidth=0.5); ax.tick_params(which="minor", length=0)
plt.colorbar(im, ax=ax, label="Attribution (per occurrence, signed)", shrink=0.8)
plt.tight_layout()
plt.savefig(output_dir / "kozak_heatmap_occurrence.png", dpi=150)
plt.show()


## 7b. CDS Codon Channel-wise Score

Per-codon attribution along the CDS (64 channels, sorted AAA..TTT).
Two views:
  (i) heatmap: codon (rows, mean-|attr| sorted desc) x position (columns)
  (ii) top-N codon mean profile lines
ggplot-ready TSV with amino-acid metadata and global rank.


In [ ]:
# CDS codon channel-wise score (per-codon positional attribution).
# Mirrors Section 7 (UTR base channel-wise) for CDS codon space.
# Aggregation matches plot/ggplot conventions: mean over transcripts, mask>0.5 drops padding.
from duet.data.utils import CODON_TO_AA

cds_ch   = nt_attr_shap["cds_codon_attr"]    # (N, cds_len, 64), signed
cds_mask = scores_shap["cds_mask"]           # (N, cds_len)

# Channel order = sorted CODON_CODES (AAA..TTT).
codons = sorted(CODON_TO_AA.keys())
assert len(codons) == cds_ch.shape[-1], f"codon count mismatch: {len(codons)} vs {cds_ch.shape[-1]}"
aa_of  = [CODON_TO_AA[c] for c in codons]

# average each codon channel ONLY over transcripts where
# that codon is present (argmax|attr|), padding excluded -> removes AAA composition bias.
L_cds = cds_ch.shape[1]
present_c = np.abs(cds_ch).argmax(-1)                       # (N, L) present codon
cds_mean = np.full((L_cds, cds_ch.shape[-1]), np.nan)
for _k in range(cds_ch.shape[-1]):
    _sel = (present_c == _k) & cds_mask
    _cnt = _sel.sum(0)
    with np.errstate(invalid="ignore"):
        _s = np.where(_sel, cds_ch[:, :, _k], 0.0).sum(0)
        cds_mean[:, _k] = np.where(_cnt >= 10, _s / np.maximum(_cnt, 1), np.nan)

# Per-codon global magnitude (mean |attr| across valid positions) -> rank codons.
codon_abs_mean = np.nanmean(np.abs(cds_mean), axis=0)                            # (64,)
codon_order_desc = np.argsort(-codon_abs_mean)                                   # high -> low
codon_rank = np.empty(64, dtype=int)
codon_rank[codon_order_desc] = np.arange(1, 65)

# --- (i) Heatmap: codon (rows, sorted by |attr| desc) x position ---
fig, ax = plt.subplots(figsize=(14, 10))
heat = cds_mean[:, codon_order_desc].T                                            # (64, L)
vmax = np.nanpercentile(np.abs(heat), 99)
im = ax.imshow(heat, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
               interpolation="nearest")
ax.set_yticks(np.arange(64))
ax.set_yticklabels([f"{codons[i]} ({aa_of[i]})" for i in codon_order_desc], fontsize=7)
ax.set_xlabel("CDS codon position")
ax.set_ylabel("Codon (sorted by mean |attribution|, desc)")
ax.set_title("CDS Codon Channel-wise Attribution (DeepLiftShap)")
plt.colorbar(im, ax=ax, label="Attribution (signed)")
plt.tight_layout()
plt.savefig(output_dir / "cds_codon_heatmap_shap.png", dpi=150)
plt.show()

# --- (ii) Top-3 / Bottom-3 codon mean profile (line plot) ---
# SIGNED selection, stop codons excluded, coverage-filtered so lines
# are continuous. Stop codons (TAA/TAG/TGA) rank high by |attr| but barely occur in
# mid-CDS, giving broken lines; we drop them and require dense positional coverage.
STOP_CODONS = {"TAA", "TAG", "TGA"}
MIN_POS = 300   # codon must have >=10 occurrences in at least this many positions
codon_signed_mean = np.nanmean(cds_mean, axis=0)                 # (64,) signed
codon_cov = (np.array([( (present_c == _k) & cds_mask ).sum(0) for _k in range(64)]) >= 10).sum(1)  # (64,) #positions
eligible = np.array([(codons[_k] not in STOP_CODONS) and (codon_cov[_k] >= MIN_POS)
                     for _k in range(64)])
_order = np.argsort(codon_signed_mean)                           # ascending (most negative first)
_order = [k for k in _order if eligible[k]]
bottom3 = _order[:3]                                             # most TE-repressing
top3 = _order[-3:][::-1]                                         # most TE-promoting

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(1, L_cds + 1)
sel_codons = [(k, "top", c) for k, c in zip(top3, ["#08519c", "#3182bd", "#6baed6"])] + \
             [(k, "bottom", c) for k, c in zip(bottom3, ["#a50f15", "#de2d26", "#fb6a4a"])]
for k, grp, color in sel_codons:
    label = f"{codons[k]} ({aa_of[k]}) {'+' if grp=='top' else '-'}"
    profile = cds_mean[:, k]
    valid = ~np.isnan(profile)
    if valid.sum() > 5:
        window = min(21, int(valid.sum()) // 2 * 2 - 1)
        if window >= 5:
            sm = np.full_like(profile, np.nan)
            sm[valid] = savgol_filter(profile[valid], window_length=window, polyorder=3)
            ax.plot(x, sm, linewidth=1.6, color=color, label=label, alpha=0.9)
        else:
            ax.plot(x, profile, linewidth=1.6, color=color, label=label, alpha=0.9)
    else:
        ax.plot(x, profile, linewidth=1.6, color=color, label=label, alpha=0.9)
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("CDS codon position")
ax.set_ylabel("Attribution (signed, per occurrence)")
ax.set_title("CDS codons: top-3 (TE-promoting) & bottom-3 (TE-repressing), "
             "stop codons excluded")
ax.legend(loc="best", fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "cds_codon_top_profile_shap.png", dpi=150)
plt.show()
print("top3 (promoting):", [f"{codons[k]}({aa_of[k]}) {codon_signed_mean[k]:+.5f}" for k in top3])
print("bottom3 (repressing):", [f"{codons[k]}({aa_of[k]}) {codon_signed_mean[k]:+.5f}" for k in bottom3])

# --- ggplot-ready TSV with amino-acid + rank metadata ---
# Companion to cell-29 `ggplot_cds_codon_profile.tsv` (raw long form);
# this one adds amino_acid + mean_abs_attr + rank so R can facet/sort easily.
rows = []
for j, codon in enumerate(codons):
    aa = aa_of[j]
    rk = int(codon_rank[j])
    mabs = float(codon_abs_mean[j])
    for p in range(L_cds):
        rows.append({
            "region": "cds",
            "position": p + 1,
            "codon": codon,
            "amino_acid": aa,
            "rank_abs": rk,
            "mean_abs_attr": mabs,
            "attribution": cds_mean[p, j],
        })
df_cds_codon_meta = pd.DataFrame(rows)
df_cds_codon_meta.to_csv(output_dir / "ggplot_cds_codon_score.tsv", sep="\t", index=False)
print("saved ggplot_cds_codon_score.tsv", df_cds_codon_meta.shape)

# Compact codon-level summary (one row per codon) for quick R bar/dot plots.
df_codon_summary = pd.DataFrame({
    "codon": codons,
    "amino_acid": aa_of,
    "mean_abs_attr": codon_abs_mean,
    "rank_abs": codon_rank,
}).sort_values("rank_abs").reset_index(drop=True)
df_codon_summary.to_csv(output_dir / "ggplot_cds_codon_summary.tsv", sep="\t", index=False)
print("saved ggplot_cds_codon_summary.tsv", df_codon_summary.shape)




# per-codon frequency + per-occurrence sensitivity (padding excluded).
_pc = np.abs(cds_ch).argmax(-1)
_rows = []
for _k in range(cds_ch.shape[-1]):
    _sel = (_pc == _k) & cds_mask
    _a = cds_ch[:, :, _k][_sel]
    _rows.append({"codon": codons[_k], "amino_acid": aa_of[_k], "n_occurrence": int(_sel.sum()),
                  "frequency": _sel.sum() / cds_mask.sum(),
                  "per_occ_signed": float(_a.mean()) if _a.size else np.nan,
                  "per_occ_abs": float(np.abs(_a).mean()) if _a.size else np.nan,
                  "total_abs": float(np.abs(_a).sum())})
_cds_occ = pd.DataFrame(_rows)
_cds_occ["total_abs_share"] = _cds_occ["total_abs"] / _cds_occ["total_abs"].sum()
_cds_occ = _cds_occ.drop(columns="total_abs").sort_values("per_occ_abs", ascending=False).reset_index(drop=True)
_cds_occ.to_csv(output_dir / "cds_codon_occurrence_summary.tsv", sep="\t", index=False)
print("saved cds_codon_occurrence_summary.tsv")
print(_cds_occ.head(12).to_string(index=False))


## ggplot-ready exports (long-format TSV for R)


In [ ]:
# ggplot-ready exports (long format) for the figures (R/ggplot).
# Each figure -> one tidy TSV. Aggregation mirrors the plot functions exactly
# (raw mean over transcripts, padding positions dropped via mask>0.5).
# Smoothing is NOT applied here (do it in R, e.g. signal::sgolayfilt or geom_smooth).
ggplot_dir = output_dir  # attribution_oof/

# per-channel present-token mean (frequency effect removed),
# matching Section 7/7b line plots. Returns (L,C) mean and (L,C) occurrence count.
def _channel_present_mean(attr, mask, min_n=10):
    # returns per-(position,channel) mean, occurrence count, and std (for SEM bands).
    present = np.abs(attr).argmax(-1)                    # (N,L)
    L, C = attr.shape[1], attr.shape[2]
    out = np.full((L, C), np.nan)
    std = np.full((L, C), np.nan)
    cnt = np.zeros((L, C), dtype=int)
    for k in range(C):
        sel = (present == k) & mask
        c = sel.sum(0)
        vals = np.where(sel, attr[:, :, k], np.nan)      # (N,L), NaN off-token/padding
        with np.errstate(invalid="ignore"):
            m = np.nanmean(vals, axis=0)
            sd = np.nanstd(vals, axis=0)
            ok = c >= min_n
            out[:, k] = np.where(ok, m, np.nan)
            std[:, k] = np.where(ok, sd, np.nan)
        cnt[:, k] = c
    return out, cnt, std


def _pos_mean(values, mask):
    """Mean over transcripts at masked positions, NaN where position mostly padding."""
    prof = np.nanmean(np.where(mask, values, np.nan), axis=0)
    pos_ok = np.nanmean(mask.astype(float), axis=0) > POS_MIN_FRAC
    prof[~pos_ok] = np.nan
    return prof, pos_ok

# --- (1) Positional importance profile: utr/cds x norm/sum ---
rows = []
for region in ["utr", "cds"]:
    mask = scores_shap[f"{region}_mask"]
    for agg in ["norm", "sum"]:
        prof, _ = _pos_mean(scores_shap[f"{region}_{agg}"], mask)
        for p, v in enumerate(prof, start=1):
            rows.append({"region": region, "agg": agg, "position": p, "score": v})
df_pos = pd.DataFrame(rows)
df_pos.to_csv(ggplot_dir / "ggplot_positional_profile.tsv", sep="\t", index=False)
print("saved ggplot_positional_profile.tsv", df_pos.shape)

# --- (2) Nucleotide-level profile (5'UTR, signed per A/C/G/T) ---
utr_nt = nt_attr_shap["utr_nt_attr"]                 # (N, 500, 4)
utr_mask = scores_shap["utr_mask"]                   # (N, 500)
nt_mean, nt_cnt, nt_std = _channel_present_mean(utr_nt, utr_mask)   # freq-removed (500,4)
rows = []
for p in range(nt_mean.shape[0]):
    for j, nt in enumerate(["A", "G", "C", "T"]):
        _n = int(nt_cnt[p, j])
        _sem = (nt_std[p, j] / np.sqrt(_n)) if _n > 0 else np.nan
        rows.append({"region": "utr", "position": p + 1, "nucleotide": nt,
                     "attribution": nt_mean[p, j], "n_occurrence": _n,
                     "std": nt_std[p, j], "sem": _sem})
df_nt = pd.DataFrame(rows)
df_nt.to_csv(ggplot_dir / "ggplot_nucleotide_profile.tsv", sep="\t", index=False)
print("saved ggplot_nucleotide_profile.tsv", df_nt.shape)

# --- (2b) CDS codon-channel positional profile (signed per codon) ---
# CDS attribution channels are 64 codons in sorted (CODON_CODES) order AAA..TTT.
cds_ch = nt_attr_shap["cds_codon_attr"]              # (N, cds_len, 64)
cds_mask = scores_shap["cds_mask"]                   # (N, cds_len)
_codons = sorted(a + b + c for a in 'ACGT' for b in 'ACGT' for c in 'ACGT')  # 64
cds_mean, cds_cnt, cds_std = _channel_present_mean(cds_ch, cds_mask)   # freq-removed (cds_len,64)
rows = []
for p in range(cds_mean.shape[0]):
    for j, codon in enumerate(_codons):
        _n = int(cds_cnt[p, j])
        _sem = (cds_std[p, j] / np.sqrt(_n)) if _n > 0 else np.nan
        rows.append({"region": "cds", "position": p + 1, "codon": codon,
                     "attribution": cds_mean[p, j], "n_occurrence": _n,
                     "std": cds_std[p, j], "sem": _sem})
df_cds_codon = pd.DataFrame(rows)
df_cds_codon.to_csv(ggplot_dir / "ggplot_cds_codon_profile.tsv", sep="\t", index=False)
print("saved ggplot_cds_codon_profile.tsv", df_cds_codon.shape)

# --- (3) Kozak heatmap (absolute only): last 14 nt of 5'UTR, per nucleotide ---
# rel_pos: distance from AUG, -14 (5'-most) .. -1 (just before AUG).
kozak_len = 14
kozak_data = nt_mean[-kozak_len:, :]                 # (14, 4), reuse nt_mean above
rows = []
for r in range(kozak_len):
    for j, nt in enumerate(["A", "G", "C", "T"]):
        rows.append({"rel_position": r - kozak_len,   # -14 .. -1
                     "kozak_index": r + 1,             # 1 .. 14 (5'->3')
                     "nucleotide": nt,
                     "attribution": kozak_data[r, j]})
df_kozak = pd.DataFrame(rows)
df_kozak.to_csv(ggplot_dir / "ggplot_kozak_heatmap.tsv", sep="\t", index=False)
print("saved ggplot_kozak_heatmap.tsv", df_kozak.shape)

# --- (4) TE group-wise positional profile (quartiles, utr/cds, norm) ---
labels = scalar_labels(scores_shap["labels"])
n_groups = 4
quantiles = np.nanquantile(labels, np.linspace(0, 1, n_groups + 1))
group_idx = np.digitize(labels, quantiles[1:-1])
group_idx[np.isnan(labels)] = -1
group_names = [f"Q{i+1}" for i in range(n_groups)]
group_ranges = {f"Q{i+1}": f"{quantiles[i]:.4f}~{quantiles[i+1]:.4f}" for i in range(n_groups)}
rows = []
for region in ["utr", "cds"]:
    mask = scores_shap[f"{region}_mask"]
    values = scores_shap[f"{region}_norm"]
    pos_ok = np.nanmean(mask.astype(float), axis=0) > POS_MIN_FRAC
    for i in range(n_groups):
        g = group_idx == i
        prof = np.nanmean(np.where(mask[g], values[g], np.nan), axis=0)
        prof[~pos_ok] = np.nan
        for p, v in enumerate(prof, start=1):
            rows.append({"region": region, "agg": "norm", "te_quartile": group_names[i],
                         "te_range": group_ranges[group_names[i]], "position": p, "score": v})
df_grp = pd.DataFrame(rows)
df_grp.to_csv(ggplot_dir / "ggplot_te_group_profile.tsv", sep="\t", index=False)
print("saved ggplot_te_group_profile.tsv", df_grp.shape)

print("\nAll ggplot tables written to", ggplot_dir)



## Fair region-level importance (UTR vs CDS)


In [ ]:
# Region-level importance: UTR vs CDS (channel-count independent).
# Standard attribution-importance measure = total attribution MAGNITUDE per position,
# i.e. sum of |attr| over channels (sum_ch |attr|). This counts how much the model
# 'attended' to a position regardless of sign, so positions whose channel effects
# cancel (e.g. A up / T down) are still counted as important. (L2 norm is avoided
# because it scales with #channels: UTR=4 vs CDS=64.)
#   total   = sum over valid positions of (sum_ch |attr|)   (length-dependent)
#   density = total / valid_length                          (length-normalized; fair)
# NOTE: attribution-based; for a causal check, cross-validate with occlusion/ISM.
u_attr = attr_shap["utr_attr"]; c_attr = attr_shap["cds_attr"]    # (N,L,ch), signed
u_mask = scores_shap["utr_mask"]; c_mask = scores_shap["cds_mask"]
u_mag = np.where(u_mask, np.abs(u_attr).sum(-1), 0.0)            # sum_ch |attr|  (N,L)
c_mag = np.where(c_mask, np.abs(c_attr).sum(-1), 0.0)

u_total = u_mag.sum(1); c_total = c_mag.sum(1)                    # (N,)
u_len = u_mask.sum(1).astype(float); c_len = c_mask.sum(1).astype(float)
u_density = u_total / np.maximum(u_len, 1)
c_density = c_total / np.maximum(c_len, 1)
utr_frac = u_total / np.maximum(u_total + c_total, 1e-12)         # UTR share of total

# --- per-transcript long table (for R: distributions, boxplots, paired tests) ---
reg_long = pd.DataFrame({
    "transcript_idx": np.tile(np.arange(len(u_total)), 2),
    "region": ["utr"] * len(u_total) + ["cds"] * len(c_total),
    "total_attr":   np.concatenate([u_total, c_total]),
    "density_attr": np.concatenate([u_density, c_density]),
    "valid_length": np.concatenate([u_len, c_len]),
    "label":        np.tile(scalar_labels(scores_shap["labels"]), 2),
})
reg_long.to_csv(output_dir / "ggplot_region_importance_per_transcript.tsv", sep="\t", index=False)

# --- summary table ---
reg_summary = pd.DataFrame([
    {"region": "utr", "total_mean": u_total.mean(), "total_median": np.median(u_total),
     "density_mean": u_density.mean(), "density_median": np.median(u_density),
     "valid_length_mean": u_len.mean()},
    {"region": "cds", "total_mean": c_total.mean(), "total_median": np.median(c_total),
     "density_mean": c_density.mean(), "density_median": np.median(c_density),
     "valid_length_mean": c_len.mean()},
])
reg_summary.to_csv(output_dir / "ggplot_region_importance_summary.tsv", sep="\t", index=False)

print("saved ggplot_region_importance_per_transcript.tsv", reg_long.shape)
print("saved ggplot_region_importance_summary.tsv")
print(f"  UTR/CDS total ratio  : {u_total.mean()/c_total.mean():.2f}x  (UTR share {utr_frac.mean():.3f})")
print(f"  UTR/CDS density ratio: {u_density.mean()/c_density.mean():.2f}x  (fair, length-normalized)")


## Attention pooling analysis (DuET v5 AttentionPooling weights)
Extract per-position attention weights from each branch's AttentionPooling (leakage-free OOF: each fold model attends only its held-out transcripts). Cross-check against the attribution positional profile. NOTE: attention is a pooling weight over the BiGRU+CNN representation (context already mixed by the GRU), not a causal explanation - use as a complementary view of where the model looks, alongside attribution.


In [ ]:
# AttentionPooling weights (OOF): where the model 'looks' when pooling.
import torch.nn.functional as F

def extract_attention(fold):
    """Per-position softmax attention for UTR & CDS branches on this fold's held-out set."""
    exp_dir = f"{EXP_PREFIX}{fold:02d}"
    ck = sorted(glob(f"{exp_dir}/ckpts/*.ckpt"), key=getmtime)[-1]
    fcfg, fdict = load_cfgs([f"{exp_dir}/config.yaml"], {})
    mdl = Module.load_from_checkpoint(ck, cfg=fcfg, dict_cfg=fdict, strict=False).eval().cuda().model
    cap = {}
    def _mk(n):
        def _h(mod, i, o): cap[n] = o.detach()
        return _h
    hu = mdl.utr_branch.attn_pool.attention.register_forward_hook(_mk('utr'))
    hc = mdl.cds_branch.attn_pool.attention.register_forward_hook(_mk('cds'))
    loader = DataLoader(torch.utils.data.Subset(full_dataset, fold_test_idx[fold]),
                        batch_size=32, shuffle=False, num_workers=1)
    wu, wc = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: (v.cuda() if isinstance(v, torch.Tensor) else v) for k, v in b.items()}
            mdl.predict(b)
            for name, store in [('utr', wu), ('cds', wc)]:
                sc = cap[name].squeeze(-1)
                mask = b[name].float().sum(-1) > 0
                w = torch.nan_to_num(F.softmax(sc.masked_fill(~mask, float('-inf')), dim=1), nan=0.0)
                store.append(w.cpu().numpy())
    hu.remove(); hc.remove(); del mdl; torch.cuda.empty_cache()
    return np.concatenate(wu), np.concatenate(wc)

_au, _ac = [], []
for f in range(N_FOLDS):
    print(f"--- attention fold {f} ---")
    u, c = extract_attention(f)
    _au.append(u); _ac.append(c)
attn_utr = np.concatenate(_au)  # (N, 500)
attn_cds = np.concatenate(_ac)  # (N, 1000)
np.save(output_dir / 'attention_utr.npy', attn_utr)
np.save(output_dir / 'attention_cds.npy', attn_cds)
print('attention UTR:', attn_utr.shape, 'CDS:', attn_cds.shape)


In [ ]:
# --- Attention visualization: UTR + CDS mean profiles & heatmaps ---
utr_mask = scores_shap['utr_mask']; cds_mask = scores_shap['cds_mask']
te = scalar_labels(scores_shap['labels']); order = np.argsort(te)

def _prof(attn, mask):
    pos_ok = np.nanmean(mask.astype(float), axis=0) > POS_MIN_FRAC
    pr = np.nanmean(np.where(mask, attn, np.nan), axis=0); pr[~pos_ok] = np.nan
    return pr
attn_prof_u = _prof(attn_utr, utr_mask)
attn_prof_c = _prof(attn_cds, cds_mask)

fig, axes = plt.subplots(4, 1, figsize=(14, 14),
                         gridspec_kw={'height_ratios': [1, 1.6, 1, 1.6]})
# (1) UTR mean profile
xu = np.arange(1, 501)
axes[0].plot(xu, attn_prof_u, color='purple', lw=1.5)
axes[0].axvspan(500 - 9 + 1, 500, alpha=0.15, color='skyblue', label='Kozak')
axes[0].set_title("5' UTR mean AttentionPooling weight (OOF)")
axes[0].set_ylabel('mean attention'); axes[0].legend(); axes[0].grid(alpha=0.3)
# (2) UTR heatmap (last 100 nt = start-proximal)
hmu = attn_utr[order][:, -100:]
im0 = axes[1].imshow(hmu, aspect='auto', cmap='magma', interpolation='nearest',
                     extent=[500 - 100 + 1, 500, len(order), 0], vmax=np.nanpercentile(hmu, 99))
axes[1].set_xlabel('5\'UTR position (last 100 nt, near AUG)')
axes[1].set_ylabel('transcripts (TE asc \u2192 desc)')
axes[1].set_title("5' UTR attention heatmap (start-proximal 100 nt, TE-sorted)")
plt.colorbar(im0, ax=axes[1], label='attention', shrink=0.8)
# (3) CDS mean profile
xc = np.arange(1, attn_cds.shape[1] + 1)
axes[2].plot(xc, attn_prof_c, color='coral', lw=1.2)
axes[2].set_title('CDS mean AttentionPooling weight (OOF)')
axes[2].set_ylabel('mean attention'); axes[2].set_xlabel('CDS codon position'); axes[2].grid(alpha=0.3)
# (4) CDS heatmap (first 100 codon = start-proximal)
hmc = attn_cds[order][:, :100]
im1 = axes[3].imshow(hmc, aspect='auto', cmap='magma', interpolation='nearest',
                     extent=[1, 100, len(order), 0], vmax=np.nanpercentile(hmc, 99))
axes[3].set_xlabel('CDS codon position (first 100, after start)')
axes[3].set_ylabel('transcripts (TE asc \u2192 desc)')
axes[3].set_title('CDS attention heatmap (start-proximal 100 codon, TE-sorted)')
plt.colorbar(im1, ax=axes[3], label='attention', shrink=0.8)
plt.tight_layout()
plt.savefig(output_dir / 'attention_analysis.png', dpi=140)
plt.show()
print(f"UTR attention last50/first50: {np.nanmean(attn_prof_u[-50:])/np.nanmean(attn_prof_u[:50]):.1f}x")
print(f"CDS attention first50/last50: {np.nanmean(attn_prof_c[:50])/np.nanmean(attn_prof_c[-50:]):.1f}x")



In [ ]:
# --- 5' UTR attention heatmap only, full canvas (same size as combined figure) ---
te = scalar_labels(scores_shap['labels']); order = np.argsort(te)
hmu = attn_utr[order][:, -100:]
fig, ax = plt.subplots(figsize=(14, 14))
im = ax.imshow(hmu, aspect='auto', cmap='magma', interpolation='nearest',
               extent=[500 - 100 + 1, 500, len(order), 0], vmax=np.nanpercentile(hmu, 99))
ax.set_xlabel("5'UTR position (last 100 nt, near AUG)")
ax.set_ylabel('transcripts (TE asc \u2192 desc)')
ax.set_title("5' UTR AttentionPooling heatmap (start-proximal 100 nt, TE-sorted)")
plt.colorbar(im, ax=ax, label='attention', shrink=0.8)
plt.tight_layout()
plt.savefig(output_dir / 'attention_utr_heatmap.png', dpi=140)
plt.show()


## Attention heatmap data as transcript x position matrix (R-ready)
Export the attention heatmap as a wide matrix (rows = TE-sorted transcripts, columns = positions; leading `txID`, `te` columns). Padding positions are NA. `HEATMAP_REGION_ONLY=False` exports all positions (UTR 500 / CDS 1000); True keeps the start-proximal window shown in the figures. Far smaller than long format and loads directly as a matrix in R.


In [ ]:
# Attention heatmap data -> transcript x position matrix (R-ready).
# Wide matrix: rows = TE-sorted transcripts, columns = positions. Two leading
# id columns (txID, te) then one column per position. Padding positions are NaN.
# HEATMAP_REGION_ONLY=True -> start-proximal window (UTR last 100 nt, CDS first
# 100 codon, as plotted); False -> ALL positions (UTR 500, CDS 1000).
HEATMAP_REGION_ONLY = False
ggplot_dir = output_dir

_te = np.asarray(scalar_labels(scores_shap['labels']))
_order = np.argsort(_te)                       # TE-sorted (rows match heatmaps)
_oof = np.concatenate(fold_test_idx)
_txid_all = (full_dataset.data['txID'].values if hasattr(full_dataset, 'data')
             else full_dataset.dataset.data['txID'].values)
_txid = _txid_all[_oof]

def _attn_matrix(attn, mask, lo, hi, pos_label):
    """Return DataFrame: txID, te, <pos_label>_<p> ... (TE-sorted rows).
    Padding positions set to NaN so R sees gaps explicitly."""
    a = np.where(mask, attn, np.nan)[_order][:, lo:hi]
    pos1 = np.arange(lo, hi) + 1                # 1-based position labels
    cols = [f'{pos_label}_{p}' for p in pos1]
    df = pd.DataFrame(a, columns=cols)
    df.insert(0, 'te', _te[_order])
    df.insert(0, 'txID', _txid[_order])
    return df

_Lu = attn_utr.shape[1]; _Lc = attn_cds.shape[1]
if HEATMAP_REGION_ONLY:
    u_lo, u_hi, c_lo, c_hi = _Lu - 100, _Lu, 0, 100
else:
    u_lo, u_hi, c_lo, c_hi = 0, _Lu, 0, _Lc

mat_u = _attn_matrix(attn_utr, utr_mask, u_lo, u_hi, 'utr')
mat_c = _attn_matrix(attn_cds, cds_mask, c_lo, c_hi, 'cds')
tag = 'region' if HEATMAP_REGION_ONLY else 'full'
pu = ggplot_dir / f'attention_utr_heatmap_matrix_{tag}.tsv.gz'
pc = ggplot_dir / f'attention_cds_heatmap_matrix_{tag}.tsv.gz'
mat_u.to_csv(pu, sep='\t', index=False, compression='gzip', na_rep='NA')
mat_c.to_csv(pc, sep='\t', index=False, compression='gzip', na_rep='NA')
print(f'UTR matrix: {mat_u.shape[0]} x {mat_u.shape[1]-2} positions -> {pu.name}')
print(f'CDS matrix: {mat_c.shape[0]} x {mat_c.shape[1]-2} positions -> {pc.name}')
# R: m <- read.table(gzfile('..._matrix_full.tsv.gz'), header=TRUE, sep='\t',
#                     check.names=FALSE); mat <- as.matrix(m[, -(1:2)])


## Attention-guided motif finding (local-peak)
Find positions where attention spikes **above its local neighborhood** (rolling z-score), independent of cross-transcript scale or global position bias - the local window is the baseline, so no global background subtraction is needed. Export peak-centered k-mers vs position-matched control as FASTA for STREME (run separately in the `motif` env). Cross-check against modisco/XSTREME motifs.


In [ ]:
# Attention local-peak motif export (rolling z-score).
from scipy.ndimage import uniform_filter1d

# rebuild UTR sequence (argmax of OOF one-hot; same order as attn_utr / attribution)
oof_order = np.concatenate(fold_test_idx)
B2 = np.array(list('AGCT'))
utr_seq = np.full((len(oof_order), 500), 'N', dtype='<U1')
for row, gidx in enumerate(oof_order):
    u = full_dataset[int(gidx)]['utr']
    u = u.numpy() if hasattr(u, 'numpy') else np.asarray(u)
    if u.shape[0] == 4 and u.shape[-1] != 4: u = u.T
    valid = u[:, :4].sum(-1) > 0
    utr_seq[row, valid] = B2[u[valid, :4].argmax(-1)]
umask = scores_shap['utr_mask']

# rolling local z-score (NaN-aware): peak = attention above its local neighborhood
W = 20  # half-window; context = 2W+1 = 41 nt
Z_THR = 2.5
EDGE = 490  # exclude the trivially-high AUG-adjacent end
_a = np.where(umask, attn_utr, 0.0); _m = umask.astype(float)
_cnt = uniform_filter1d(_m, 2*W+1, axis=1, mode='nearest')
_mean = uniform_filter1d(_a, 2*W+1, axis=1, mode='nearest') / np.clip(_cnt, 1e-6, None)
_var = uniform_filter1d(_a**2, 2*W+1, axis=1, mode='nearest') / np.clip(_cnt, 1e-6, None) - _mean**2
lz = (np.where(umask, attn_utr, np.nan) - _mean) / np.sqrt(np.clip(_var, 1e-12, None))
peak = (lz > Z_THR) & umask
peak[:, EDGE:] = False
print(f'peaks: {peak.sum()} ({peak.sum(1).mean():.2f}/transcript)')

# collect peak-centered 7-mers + position-matched control (non-peak valid sites)
K = 3  # +/- 3 => 7-mer
rng = np.random.default_rng(0)
peak_seqs, ctrl_seqs = [], []
for i in range(len(utr_seq)):
    pp = np.where(peak[i])[0]
    pp = pp[(pp >= K) & (pp < 500 - K)]
    for pos in pp:
        s = ''.join(utr_seq[i, pos-K:pos+K+1])
        if 'N' not in s: peak_seqs.append(s)
    cand = np.where(umask[i] & ~peak[i] & (np.arange(500) < EDGE))[0]
    cand = cand[(cand >= K) & (cand < 500 - K)]
    for _ in range(min(len(pp), len(cand))):
        pos = rng.choice(cand); s = ''.join(utr_seq[i, pos-K:pos+K+1])
        if 'N' not in s: ctrl_seqs.append(s)

mdir = Path('motif_search/data')
def _wfasta(seqs, path):
    with open(path, 'w') as f:
        for j, s in enumerate(seqs): f.write(f'>{j}\n{s}\n')
_wfasta(peak_seqs, mdir / 'utr_attn_peak.fa')
_wfasta(ctrl_seqs, mdir / 'utr_attn_control.fa')
print(f'peak 7-mers: {len(peak_seqs)} -> motif_search/data/utr_attn_peak.fa')
print(f'control:     {len(ctrl_seqs)} -> motif_search/data/utr_attn_control.fa')
from collections import Counter
print('top peak 7-mers:', Counter(peak_seqs).most_common(8))
print('\nRun STREME separately:  conda activate motif; bash motif_search/run_utr_attention_streme.sh')


In [ ]:
# CDS attention local-peak motif export (codon-level; mirrors the 5'UTR cell above).
# Reuses attn_cds (in memory, same OOF order as attn_utr). Codon-level adaptations:
# rolling window in CODONS (W_cd), start-codon region masked (structural attention
# spike), peak-centered +/-FLANK_cd codons -> STREME k-mer.
CODONS = sorted(a + b + c for a in 'ACGT' for b in 'ACGT' for c in 'ACGT')  # == CODON_CODES order
W_cd = 21          # local window in codons
Z_cd = 2.5         # z-score peak threshold
START_SKIP = 10    # codons masked near start (strong structural spike)
FLANK_cd = 1       # +/- codons around peak -> (2*FLANK_cd+1) codons

# rebuild CDS codon indices + validity in the same OOF order as attn_cds
Mc, Lc = attn_cds.shape
cds_idx = np.zeros((Mc, Lc), dtype=int)
cds_val = np.zeros((Mc, Lc), dtype=bool)
for row, gidx in enumerate(oof_order):
    g = full_dataset[int(gidx)]['cds']
    g = g.numpy() if hasattr(g, 'numpy') else np.asarray(g)
    if g.shape[0] == 64 and g.shape[-1] != 64:
        g = g.T
    oh = g[:, :64]
    cds_idx[row] = oh.argmax(-1)
    cds_val[row] = oh.sum(-1) > 0

# rolling local z-score (codon-level)
_cmean = uniform_filter1d(attn_cds, W_cd, axis=1, mode='nearest')
_cvar = uniform_filter1d(attn_cds ** 2, W_cd, axis=1, mode='nearest') - _cmean ** 2
lz_cd = (attn_cds - _cmean) / np.sqrt(np.clip(_cvar, 1e-12, None))
peak_cd = (lz_cd > Z_cd) & cds_val
peak_cd[:, :START_SKIP] = False
peak_cd[:, Lc - FLANK_cd:] = False
print(f'CDS peaks: {peak_cd.sum()} ({peak_cd.sum(1).mean():.2f}/transcript)')

rng_cd = np.random.default_rng(42)
cds_peak_seqs, cds_ctrl_seqs = [], []
for i in range(Mc):
    pp = np.where(peak_cd[i])[0]
    pp = pp[(pp >= FLANK_cd) & (pp < Lc - FLANK_cd)]
    for pos in pp:
        if cds_val[i, pos - FLANK_cd:pos + FLANK_cd + 1].all():
            cds_peak_seqs.append(''.join(CODONS[cds_idx[i, pos + k]] for k in range(-FLANK_cd, FLANK_cd + 1)))
    cand = np.where((~peak_cd[i]) & cds_val[i])[0]
    cand = cand[(cand >= START_SKIP) & (cand >= FLANK_cd) & (cand < Lc - FLANK_cd)]
    if len(cand) and len(pp):
        pick = rng_cd.choice(cand, size=min(len(pp), len(cand)), replace=False)
        for pos in pick:
            if cds_val[i, pos - FLANK_cd:pos + FLANK_cd + 1].all():
                cds_ctrl_seqs.append(''.join(CODONS[cds_idx[i, pos + k]] for k in range(-FLANK_cd, FLANK_cd + 1)))

_wfasta(cds_peak_seqs, mdir / 'cds_attn_peak.fa')
_wfasta(cds_ctrl_seqs, mdir / 'cds_attn_control.fa')
print(f'CDS peak {2*FLANK_cd+1}-mers ({(2*FLANK_cd+1)*3} nt): {len(cds_peak_seqs)} -> motif_search/data/cds_attn_peak.fa')
print(f'CDS control: {len(cds_ctrl_seqs)} -> motif_search/data/cds_attn_control.fa')
print('\nRun STREME separately:  conda activate motif; bash motif_search/run_cds_attention_streme.sh')

## Attention-guided motif: STREME results
Parse `motif_search/results/attention_streme/streme.txt` (run separately) and show top motifs (E-value) + seqlogos. STREME output is standard ACGT alphabet (operates on the exported FASTA, not the AGCT-ordered model one-hot), so no channel reorder needed here. Cross-reference: compare with attribution-based motifs (TF-MoDISco) viewed in `motif_search/analyze_motifs_finalized.ipynb`.


In [ ]:
# Parse attention-STREME results + seqlogos.
import logomaker
streme_txt = Path('motif_search/results/utr_attention_streme/streme.txt')
if not streme_txt.exists():
    print('Run first:  conda activate motif; bash motif_search/run_utr_attention_streme.sh')
else:
    motifs = []  # (name, E, nsites, pwm DataFrame ACGT)
    lines = streme_txt.read_text().splitlines()
    i = 0
    while i < len(lines):
        if lines[i].startswith('MOTIF'):
            name = lines[i].split()[1]
            j = i + 1
            while j < len(lines) and not lines[j].startswith('letter-probability'): j += 1
            hdr = lines[j]; E = hdr.split('E=')[1].strip() if 'E=' in hdr else 'NA'
            ns = hdr.split('nsites=')[1].split()[0] if 'nsites=' in hdr else 'NA'
            rows, j = [], j + 1
            while j < len(lines) and lines[j].strip() and not lines[j].startswith('MOTIF'):
                pr = lines[j].split()
                if len(pr) >= 4: rows.append([float(x) for x in pr[:4]])
                j += 1
            motifs.append((name, E, ns, pd.DataFrame(rows, columns=['A','C','G','T'])))
            i = j
        else:
            i += 1
    print(f'{len(motifs)} STREME motifs (attention peak vs position-matched control)')
    for name, E, ns, _ in motifs[:6]:
        print(f'  {name}: E={E}  nsites={ns}')
    # seqlogos for top-6
    top = motifs[:6]
    fig, axes = plt.subplots(len(top), 1, figsize=(7, 1.4 * len(top)))
    if len(top) == 1: axes = [axes]
    for ax, (name, E, ns, pwm) in zip(axes, top):
        info = logomaker.transform_matrix(pwm, from_type='probability', to_type='information')
        logomaker.Logo(info, ax=ax, color_scheme='classic')
        ax.set_yticks([]); ax.set_xticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_ylabel(f'{name}\nE={E}', rotation=0, ha='right', va='center', fontsize=8)
    axes[0].set_title('Attention local-peak motifs (STREME, top-6 by E-value)')
    plt.tight_layout()
    plt.savefig(output_dir / 'attention_streme_logos.png', dpi=140)
    plt.show()
    print('\nNote: dominant motif is uAUG (S-ATG-N); matches TF-MoDISco neg-pattern uAUG')
    print('seen in analyze_motifs_finalized.ipynb - attention independently recovers it.')


In [ ]:
# STREME -> tidy TSV for R (ggseqlogo).
# Output:
#   ggplot_attention_streme_pwm.tsv   long: motif, name, evalue, nsites, position(1..w), nt(A/C/G/T), prob
#   ggplot_attention_streme_summary.tsv  one row per motif: motif, name, evalue, nsites, width
import re

_streme_txt = Path("motif_search/results/utr_attention_streme/streme.txt")

if not _streme_txt.exists():
    print(f"[skip] {_streme_txt} not found - run motif_search/run_utr_attention_streme.sh first")
else:
    _lines = _streme_txt.read_text().splitlines()
    _records, _summary = [], []
    i = 0
    while i < len(_lines):
        line = _lines[i]
        m = re.match(r"^MOTIF\s+(\S+)(?:\s+(\S+))?", line)
        if not m:
            i += 1; continue
        motif_id = m.group(1)                                 # e.g. "1-SATGNN"
        motif_name = m.group(2) or motif_id                   # e.g. "STREME-1"
        # next non-empty line should be the letter-probability header
        j = i + 1
        while j < len(_lines) and not _lines[j].startswith("letter-probability"):
            j += 1
        if j >= len(_lines):
            break
        hdr = _lines[j]
        ev = re.search(r"E=\s*([0-9.eE+\-]+)", hdr)
        ns = re.search(r"nsites=\s*([0-9]+)", hdr)
        w  = re.search(r"w=\s*([0-9]+)", hdr)
        evalue   = ev.group(1) if ev else "NA"
        nsites   = int(ns.group(1)) if ns else -1
        width    = int(w.group(1)) if w else -1
        # PWM rows: next `width` lines with 4 floats each.
        k = j + 1
        pwm_rows = []
        while k < len(_lines) and len(pwm_rows) < (width if width > 0 else 100):
            parts = _lines[k].split()
            if len(parts) >= 4:
                try:
                    pwm_rows.append([float(x) for x in parts[:4]])
                except ValueError:
                    break
            elif not _lines[k].strip():
                # blank line ends matrix
                if pwm_rows:
                    break
            k += 1
        for pos, probs in enumerate(pwm_rows, start=1):
            for nt, p in zip("ACGT", probs):
                _records.append({
                    "motif": motif_id, "name": motif_name,
                    "evalue": evalue, "nsites": nsites,
                    "position": pos, "nt": nt, "prob": p,
                })
        _summary.append({
            "motif": motif_id, "name": motif_name,
            "evalue": evalue, "nsites": nsites,
            "width": len(pwm_rows),
        })
        i = k

    df_streme_pwm = pd.DataFrame(_records)
    df_streme_sum = pd.DataFrame(_summary)

    pwm_path = output_dir / "ggplot_attention_streme_pwm.tsv"
    sum_path = output_dir / "ggplot_attention_streme_summary.tsv"
    df_streme_pwm.to_csv(pwm_path, sep="\t", index=False)
    df_streme_sum.to_csv(sum_path, sep="\t", index=False)
    print(f"saved {pwm_path.name}  shape={df_streme_pwm.shape}")
    print(f"saved {sum_path.name}  shape={df_streme_sum.shape}")
    print("\nR usage (ggseqlogo). motif factor in input order avoids dplyr alpha-sort mismatch.")
    print("Facet titles: 'CONSENSUS (E=...)'  (e.g. 'SATGNN (E=2.2e-717)')")
    print("  library(ggseqlogo); library(dplyr); library(tidyr); library(ggplot2)")
    print(f"  pwm <- read.table('{pwm_path}', header=TRUE, sep='\\t')")
    print(f"  smr <- read.table('{sum_path}', header=TRUE, sep='\\t', colClasses=c(evalue='character'))")
    print("  motif_order <- unique(pwm$motif)                       # STREME rank order")
    print("  pwm$motif <- factor(pwm$motif, levels=motif_order)")
    print("  pwm_list <- pwm |> group_by(motif) |> group_map(~ {")
    print("      m <- tidyr::pivot_wider(.x[,c('position','nt','prob')], names_from='position', values_from='prob')")
    print("      mat <- as.matrix(m[,-1]); rownames(mat) <- m$nt; mat")
    print("  })")
    print("  # Build 'CONSENSUS (E=...)' labels. STREME motif id is 'RANK-CONSENSUS'.")
    print("  consensus <- sub('^[0-9]+-', '', motif_order)")
    print("  evalue    <- smr$evalue[match(motif_order, smr$motif)]")
    print("  labels    <- paste0(consensus, ' (E=', evalue, ')')")
    print("  names(pwm_list) <- labels")
    print("  ggseqlogo::ggseqlogo(pwm_list[1:6], method='bits', ncol=1)")
